# Bài 3.2: Nhận diện Thực thể có Tên (NER) với `PhoNLP` & `PhoBERT`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vietnamese-nlp-mastery/vietnamese-nlp-mastery/blob/main/03_Pretrained_Transformers/02_phonlp_named_entity_recognition.ipynb)

> **Mục tiêu bài học:**
> 1. Sử dụng **`PhoNLP`** (VinAI Research) — mô hình đa nhiệm kết hợp POS Tagging, NER và Dependency Parsing dựa trên `PhoBERT`.
> 2. Giải quyết bài toán **Subword Alignment** khi Fine-tune `PhoBERT` cho bài toán Token Classification (NER theo định dạng IOB2 trên tập `vinai/phoner_covid19` và `VLSP`).

In [1]:
import json
from pathlib import Path

ner_sample_path = Path("../datasets/samples/ner_vlsp_phoner_sample.jsonl")
with open(ner_sample_path, "r", encoding="utf-8") as f:
    ner_samples = [json.loads(line) for line in f]

print("=== MẪU DỮ LIỆU NER CHUẨN IOB2 (Word-level) ===")
for word, tag in zip(ner_samples[0]["words"], ner_samples[0]["tags"]):
    print(f"  {word:<18} -> {tag}")

=== MẪU DỮ LIỆU NER CHUẨN IOB2 (Word-level) ===
  Viện               -> B-ORGANIZATION
  Nghiên_cứu         -> I-ORGANIZATION
  Trí_tuệ            -> I-ORGANIZATION
  Nhân_tạo           -> I-ORGANIZATION
  VinAI              -> B-ORGANIZATION
  tại                -> O
  Hà_Nội             -> B-LOCATION
  vừa                -> O
  công_bố            -> O
  mô_hình            -> O
  PhoBERT            -> B-MISC
  .                  -> O


## Kỹ thuật Căn chỉnh Nhãn Subword (Subword Label Alignment) cho PhoBERT NER

Khi một từ ghép dài (ví dụ `Nguyễn_Khắc_Chúc`) đi qua BPE Tokenizer của PhoBERT, nó có thể bị tách thành nhiều subword (`Nguyễn_Khắc_@@`, `Chúc`). Ta gán nhãn gốc cho subword đầu tiên của từ và gán `-100` cho các subword tiếp theo để hàm `CrossEntropyLoss` của PyTorch tự động bỏ qua:

```python
def align_labels_with_subwords(words: list[str], tags: list[int], tokenizer):
    # Nối câu đã tách từ và tokenize bằng PhoBERT
    sentence_str = " ".join(words)
    subwords = tokenizer.tokenize(sentence_str)

    aligned_labels = []
    word_idx = 0
    is_continuation = False

    for sub in subwords:
        if not is_continuation:
            aligned_labels.append(tags[word_idx])
        else:
            aligned_labels.append(-100)  # Bỏ qua trong hàm loss

        # Trong PhoBERT (fairseq BPE), subword kết thúc bằng '@@' nghĩa là từ chưa kết thúc
        if sub.endswith("@@"):
            is_continuation = True
        else:
            is_continuation = False
            word_idx += 1

    return subwords, aligned_labels
```